# 機器學習：預測迴歸模型 (Regression Model)
這個 Notebook 負責載入 Feature Engineering 處理後的資料，並建構一個具備自動前處理 (Preprocessing) 特性的迴歸模型管線 (Pipeline)。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 視覺化繪圖設定
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 支援中文顯示
plt.rcParams['axes.unicode_minus'] = False

MemoryError: 

In [ ]:
# 1. 載入已經處理好的特徵資料
data_path = '../data/processed/features_ready.csv'
df = pd.read_csv(data_path)

print(f"載入的資料夾維度: {df.shape}")
display(df.head(3))

In [ ]:
# 2. 定義預測目標 (Y) 與特徵矩陣 (X)
target_col = 'reservation_type'  # <-- 依據您的要求，指定目標欄位

# 防呆檢查：如果資料內目前沒有這個欄位，暫時用 'lead_time' 當作測試目標，讓程式可以順利跑起來
if target_col not in df.columns:
    print(f"⚠️ 警告：在特徵表中找不到 '{target_col}'。請確認之前是否已產出此欄位。目前暫以 'lead_time' 示範迴歸預測！")
    target_col = 'lead_time'

# 移除所有目標欄位為空值的無效列
df = df.dropna(subset=[target_col])

# 其餘欄位全部當作 X 特徵
X = df.drop(columns=[target_col])
y = df[target_col]

# 安全機制：移除不能直接被迴歸模型丟入的「字串時間」欄位 (日期必須轉為數值，這裡直接從 X 剃除)
cols_to_exclude = ['booking_time', 'reservation_time', 'member_birthdate']
X = X.drop(columns=[c for c in cols_to_exclude if c in X.columns], errors='ignore')

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

In [ ]:
# 3. 自動區分數值與類別特徵，並建立對應的前處理 Pipeline
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 數值型特徵：中位數補值 -> 標準化
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 類別型特徵：補 missing 標籤 -> One-Hot-Encoding 轉虛擬變數
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 打包在一起成為完整的 Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# 4. 切分訓練集 (Training Set) 與測試集 (Test Set)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {X_train.shape[0]}")
print(f"Test instances: {X_test.shape[0]}")

In [ ]:
# 5. 建立完整的機器學習 Pipeline 並進行訓練模型
# 這裡採用了適合處理異質資料且不需要複雜調參的「隨機森林」(Random Forest Regressor) 示範
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model... (這可能會需要幾分鐘的時間，請稍候)")
model_pipeline.fit(X_train, y_train)
print("Training completed!")

In [ ]:
# 6. 對測試集進行預測並輸出成效評估 (Evaluation Results)
y_pred = model_pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== Regression Model Performance ===")
print(f"RMSE (均方根誤差): {rmse:.4f}")
print(f"MAE  (絕對平均誤差): {mae:.4f}")
print(f"R^2  (判定係數): {r2:.4f}  <-- 越接近 1 越好")

In [ ]:
# 7. 視覺化模型誤差：真實值 vs 預測值分布圖
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5, color='royalblue')

# 畫出 y=x 的完美預測基準線
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)

plt.title('True vs Predicted Output')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.tight_layout()
plt.show()